# 6. Constraint at core-motif binding sites and on their readers (Fig. 3c-h)

* **c, d, e**: phyloP100way conservation at the core motifs' binding sites in each of the 116 biosamples' promoter windows, against ten random sets of 43 JASPAR control motifs (`code/conservation_sequence/`);
* **f, g**: gnomAD constraint of the reader transcription factors and of their DNA-binding domains (`code/conservation_protein/`);
* **h**: the reader constraint per motif family.

Panels a and b are summaries of the MEME files and are not covered here.

Inputs: Zenodo `fig3_conservation_inputs.tar.gz` under `$ENHANCER_ROOT` — the phyloP bigWig restricted to the promoter windows, the K562 FIMO scans, the per-biosample phyloP summaries of all 116 biosamples, and the UniProt and gnomAD tables.

In [ ]:
# ---- Shared setup (identical in every notebook) --------------------------------------------------------
import os, sys, json, subprocess, time, glob, shutil
REPO = os.path.abspath(os.path.join(os.getcwd(), ".."))          # notebooks/ lives one level below the repository root
CODE = os.path.join(REPO, "code")
roots = json.load(open(os.path.join(REPO, ".roots.json")))          # written by bin/setup_work_dir.py (see PATHS.md)
ENHANCER_ROOT, SEQ_ROOT = roots["enhancer_root"], roots["seq_root"]
os.environ["ENHANCER_ROOT"], os.environ["SEQ_ROOT"] = ENHANCER_ROOT, SEQ_ROOT   # the scripts in code/ read their data root from here
def run(cmd, cwd=None, env=None):
    """Run one repository script from the right folder and stream its output here. Stops the notebook if it fails."""
    print("$", cmd, "   (in", cwd or os.getcwd(), ")"); t = time.time()
    if cmd.startswith("python "): cmd = f'"{sys.executable}" ' + cmd[7:]   # the interpreter this notebook runs on, not whatever comes first on PATH
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, executable="/bin/bash", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    for line in p.stdout: print(line, end="")
    p.wait(); print(f"[exit {p.returncode}, {time.time()-t:.0f} s]")
    assert p.returncode == 0, "this step failed - read the output above"
def n_motifs(meme):
    """Number of MOTIF entries in a MEME file."""
    return sum(1 for l in open(meme) if l.startswith("MOTIF")) if os.path.exists(meme) else None
def show_logos(meme, n=5, title=""):
    """Sequence logos of the first n motifs of a MEME file (logomaker)."""
    import numpy as np, pandas as pd, matplotlib.pyplot as plt, logomaker
    blocks = open(meme).read().split("MOTIF ")[1:n+1]; fig, axes = plt.subplots(1, len(blocks), figsize=(2.2*len(blocks), 1.4))
    for ax, b in zip(np.atleast_1d(axes), blocks):
        rows = [l.split() for l in b.split("\n")[2:] if l.strip() and not l.startswith("letter") and not l.startswith("URL")]
        pwm = pd.DataFrame([[float(x) for x in r] for r in rows if len(r) == 4], columns=list("ACGT"))
        logomaker.Logo(logomaker.transform_matrix(pwm, from_type="probability", to_type="information"), ax=ax); ax.set_title(b.split("\n")[0], fontsize=8); ax.set_yticks([]); ax.set_xticks([])
    fig.suptitle(title, fontsize=9); plt.tight_layout(); plt.show()
print("repository:", REPO); print("ENHANCER_ROOT:", ENHANCER_ROOT); print("SEQ_ROOT:", SEQ_ROOT)


In [ ]:
# ---- CONFIGURATION ---------------------------------------------------------------------------------------------
BSSID = "BSS00762"
RUN_PHYLOP_FOR_BSSID = True     # part 1.1: score every core/control-pool site of BSSID against the phyloP subset (about 12 min); False: skip
CS = f"{CODE}/conservation_sequence"; CP = f"{CODE}/conservation_protein"; PH = f"{CODE}/posthoc_analysis"; F3 = f"{REPO}/figures/fig3"
BW = f"{CS}/hg38.phyloP100way.promoter_windows.bw"       # from the Zenodo archive; the full UCSC hg38.phyloP100way.bw works the same way
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import Image, display
for k, p in {"phyloP subset bigWig": BW, "K562 pool scan": f"{PH}/results/pool_scan/{BSSID}_fimo.txt.gz", "K562 window coordinates": f"{ENHANCER_ROOT}/data_generated/{BSSID}.coord.exp.df.final.csv",
             "per-biosample phyloP summaries": f"{CS}/results/pool_conservation", "UniProt": f"{CP}/uniprot_sprot_human.dat.gz", "gnomAD RMC": f"{CP}/gnomAD_v2.1.1_transcripts_with_rmc.tsv"}.items():
    print(f"  {'ok ' if os.path.exists(p) else '-- '} {k}: {p}")
print("biosamples with phyloP summaries:", len(glob.glob(f"{CS}/results/pool_conservation/*_motif_summary.tsv")), "+ extra", len(glob.glob(f"{CS}/results/pool_conservation_extra/*_motif_summary.tsv")))

## 6.1 One biosample: phyloP at every binding site (the per-biosample step behind c-e)
`05_pool_conservation.py --bssid B --bigwig <bigWig>` reads the FIMO sites of the core motifs and of the control scan pool in that biosample's promoter windows, maps them to genome coordinates, averages phyloP over each site, keeps the best site per gene and motif, and writes the per-motif median and mean -> `results/pool_conservation/{B}_motif_summary.tsv`. About 12 min for K562; the result is compared with the shipped summary.

In [ ]:
if RUN_PHYLOP_FOR_BSSID:
    run(f"python 05_pool_conservation.py --bssid {BSSID} --bigwig {BW} --outdir results/pool_conservation_demo", cwd=CS)
    run(f"python 05_pool_conservation.py --bssid {BSSID} --bigwig {BW} --scan {PH}/results/pool_scan_extra/{BSSID}_fimo.txt.gz --outdir results/pool_conservation_extra_demo", cwd=CS)
    new = pd.read_csv(f"{CS}/results/pool_conservation_demo/{BSSID}_motif_summary.tsv", sep="\t").set_index("motif_id"); old = pd.read_csv(f"{CS}/results/pool_conservation/{BSSID}_motif_summary.tsv", sep="\t").set_index("motif_id")
    j = new.join(old, rsuffix="_shipped"); print(f"{len(j)} motifs | max |median phyloP difference| vs the shipped summary: {np.abs(j.median_score - j.median_score_shipped).max():.2e}")
    print(j[["group", "n_genes", "median_score", "median_score_shipped"]].head(6).to_string())
    core, ctrl = j[j.group == "core"].median_score, j[j.group != "core"].median_score; print(f"{BSSID}: median phyloP of the best site per gene, core motifs {core.median():.3f} vs pool motifs {ctrl.median():.3f}")

## 6.2 All 116 biosamples: panels c, d, e
`07_oldpool_10sets.py` reads the per-biosample summaries, draws the ten control sets from the pool, and writes the panel tables to `results/oldpool10/`.

In [ ]:
run("python 07_oldpool_10sets.py", cwd=CS)
st = pd.read_csv(f"{CS}/results/oldpool10/fig3_stats.csv"); print(st[["set", "n_ctrl", "core_median", "ctrl_median", "diff", "p_fig3c", "cg_adjusted_advantage", "p_fig3d", "n_core_wins", "p_fig3e"]].round(4).to_string(index=False))

## 6.3 The reader proteins: panels f, g, h
* `01_protein_conservation.py`: gnomAD gene constraint of the reader TFs against all other human TFs;
* `02_dbd_conservation.py`: regional missense constraint of their UniProt-annotated DNA-binding domains (panel g);
* `figures/fig3/code/03_family_and_extended_constraint.py` (the panel f table) and `07_family_reader_constraint.py` (panel h).

In [ ]:
run("python 01_protein_conservation.py", cwd=CP); run("python 02_dbd_conservation.py", cwd=CP)
for s in ("03_family_and_extended_constraint.py", "07_family_reader_constraint.py"): run(f"python {s}", cwd=f"{F3}/code")
print(pd.read_csv(f"{F3}/code/results/constraint_comparison_extended.csv").round(3).to_string(index=False))

## 6.4 The panels
`make_fig3_panels.py` draws every panel from the tables above (a and b from the shipped motif summaries) and the assembled figure.

In [ ]:
run("python make_fig3_panels.py", cwd=F3)
for p in ("fig3c_conservation", "fig3d_cg_adjusted", "fig3e_biosample_advantage", "fig3f_protein_constraint", "fig3g_dbd", "fig3h_family_conservation"): display(Image(f"{F3}/{p}.png", width=260))